# 3. Generate Embeddings

*Goal: Convert string data (Command Lines) into dense vector representations.*

In this notebook, we move from raw text to machine-readable numbers. An embedding is a list of floating-point numbers (a vector) that captures the semantic meaning of text. By converting our command lines into embeddings, we can use math to figure out which commands are similar to each other.

## Step 1 — Import Libraries

We will need to import two libraries for this notebook:
1. `time` module for throttling API calls.
2. `pandas` for our DataFrame operations.
3. `SecretStr` from `pydantic`, which is a utility that prevents secrets from accidentally being printed to the console or saved in notebook output.

In [ ]:
import time  # <--- Time module for throttling API calls
import pandas as pd  # <--- DataFrame usage for dataset operations
from pydantic import SecretStr  # <--- Masked string

## Step 2 — Load the Dataset

Let's pick up right where we left off. We need to load the Parquet file we generated at the end of the last notebook.

Use `pd.read_parquet()` to load `"_02_evtx_commands.parquet"` into a variable called `dataframe`.

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_parquet.html

In [ ]:
dataframe = pd.read_parquet("_02_evtx_commands.parquet")

## Step 3 — Initialize the Embeddings Client
Create an embeddings client and assign it to the `embeddings_client` variable. We will being using LangChain for the majority of our LLM logic. The nice thing about using LangChain is that it makes it trivial to adjust AI providers. Depending on the provider you are using, import the appropriate Chat class.


- OpenAI `from langchain_openai import OpenAIEmbeddings`
- Google AI Studio `from langchain_google_genai import GoogleGenerativeAIEmbeddings`
- Ollama `from langchain_ollama import OllamaEmbeddings`

Both OpenAI and Google will require API keys, but if you are running Ollama locally, you will not need an API key. Here are examples of creating a chat client for these providers.

```python
OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=SecretStr(api_key)
)
```

```python
GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key=SecretStr(api_key)
)
```

```python
OllamaEmbeddings(
    model="embeddinggemma"
)
```

Related Docs:
- https://docs.langchain.com/oss/python/integrations/embeddings/openai
- https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai
- https://docs.langchain.com/oss/python/integrations/embeddings/ollama

In [ ]:
from langchain_openai import OpenAIEmbeddings

api_key = input("Enter your Azure OpenAI API key: ")

embeddings_client = OpenAIEmbeddings(
    model="text-embedding-3-large",
    api_key=SecretStr(api_key)
)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

api_key = input("Enter your Google AI Studio API key: ")

embeddings_client = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key=SecretStr(api_key),
    task_type="RETRIEVAL_DOCUMENT"
)

In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings_client = OllamaEmbeddings(
    model="embeddinggemma"
)

## Step 4 — Extract Unique Command Lines

API calls cost time and money. If the command `whoami` appears 5,000 times in our dataset, we shouldn't creat 5,000 embeddings for it. We should embed it *once* and map it back later.

1. Isolate the `CommandLine` column/series in your DataFrame.
2. Chain the `.unique()` method to get only the distinct values (this returns an numpy array)
3. Chain the `.tolist()` method to convert the numpy array result into a standard Python list.
4. Save this list to a variable called `unique_commands`.

Example:
```python
dataframe["SomeColumn"].unique().tolist()
```

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.unique.html
 - https://numpy.org/doc/stable/reference/generated/numpy.ndarray.tolist.html#numpy.ndarray.tolist


In [ ]:
unique_commands = dataframe["CommandLine"].unique().tolist()

## Step 5 — Generate Embeddings

Now we will actually generate the embeddings. Because we might have thousands of unique commands, sending them all to an API at once might trigger a "Rate Limit Exceeded" error. To prevent this, we've provided a custom batching loop below. It processes the commands in chunks of 250 and pauses if it goes too fast.

Create the `generate_embeddings` function below so we can use it for embedding generation.

In [ ]:
def generate_embeddings(
        _embedding_client,
        _unique_commands: list[str]
) -> list[list[float]]:
    _batch_size = 250
    _unique_embeddings = []

    for _i in range(0, len(_unique_commands), _batch_size):
        _batch = _unique_commands[_i:_i + _batch_size]
        _batch_num = _i // _batch_size + 1
        _total_batches = (len(_unique_commands) + _batch_size - 1) // _batch_size
        print(f"Processing batch {_batch_num}/{_total_batches} ({len(_batch)} commands)...")
        _start = time.time()
        _unique_embeddings.extend(_embedding_client.embed_documents(_batch))
        _elapsed = time.time() - _start
        if _i + _batch_size < len(_unique_commands):
            _wait = max(0, 1 - _elapsed)
            print(f"  Batch done in {_elapsed:.1f}s. Waiting {_wait:.1f}s before next batch...")
            time.sleep(_wait)

        print(f"Done. Total embeddings: {len(_unique_embeddings)}")

    return _unique_embeddings

Generate embeddings by calling the `generate_embeddings` function and passing it the embeddings client and unique commands. Assign the output to a variable called `unique_embeddings`.

In [ ]:
unique_embeddings = generate_embeddings(embeddings_client, unique_commands)

If you want to see how many dimentions your embeddings have, print the length of the first item in the `unique_embeddings` list.

In [ ]:
print(f"Number of dimensions: {len(unique_embeddings[0])}")

## Step 6 — Build a Command-to-Embedding Mapping

We now have two lists of the exact same length: `unique_commands` (the text) and `unique_embeddings` (the vectors). We need to link them together into a lookup table where commands are the keys, and embeddings are the values.

1. Use Python's built-in `zip()` function, passing in your commands list first, and your embeddings list second.
2. Wrap the `zip(...)` function in a `dict(...)` call to convert those pairs into a Python dictionary.
3. Save this dictionary to a variable named `unique_mapping`.

Example:
```python
dictionary_mapping = dict(zip(keys, values))
```

Related Docs:
 - https://docs.python.org/3/library/functions.html#zip


In [ ]:
unique_mapping = dict(zip(unique_commands, unique_embeddings))

## Step 7 — Add Embeddings to the DataFrame

Now that we have our lookup table, we can easily fan the embeddings back out to the full dataset. We will use Pandas' vectorized `.map()` function.

1. Create a new column in your DataFrame called `"Embedding"`.
2. Set it equal to `dataframe["CommandLine"].map(...)`, passing your `unique_mapping` dictionary into the `.map()` method.
3. Display the first few rows using `.head()` to verify the new column exists and contains lists of floats.

> Because `.map()` uses the dictionary keys as a lookup, rows with identical command lines automatically share the same embedding vector.

Example:
```python
dataframe["MAPPED VALUES"] = dataframe["COLUMN TO MAP"].map(MAPPING)
```

Related Docs:
 - https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html

In [ ]:
dataframe["Embedding"] = dataframe["CommandLine"].map(unique_mapping)
dataframe.head()

## Step 8 — Save the DataFrame to Parquet

Persist the enriched DataFrame — now containing the `Embedding` column — to disk as `_03_evtx_commands_with_embeddings.parquet`.

> This file is the input for notebook **4-cluster-data**, where the embedding vectors will be grouped using unsupervised clustering algorithms.

In [ ]:
dataframe.to_parquet("_03_evtx_commands_with_embeddings.parquet")